# Heart Disease Encoder Artifacts (Notebook 1)

This is a day-0 heart-failure encoder notebook built to mirror the structure of the original mortality encoder notebook, but simplified for this experiment.

## What changed versus `01_build_artifacts`
- Label is now heart failure present during admission: `1` if the admission has any ICD-9 code starting with `428`
- Window is fixed to day 0 only
- Retrieval is removed
- Risk/day-trajectory features are removed
- Discharge summaries are not used
- A new heart-failure terminology scrubber removes explicit phrases like `heart failure`, `heart disease`, `CHF`, etc. from the note text before embeddings are created

## What this notebook does
1. Loads `ADMISSIONS.csv`, `DIAGNOSES_ICD.csv`, and `D_ICD_DIAGNOSES.csv`
2. Builds the admission-level heart-failure label from `428*`
3. Streams `NOTEEVENTS.csv` once and keeps only day-0 non-discharge notes
4. Scrubs explicit heart-failure terminology from those notes
5. Builds admission-level documents
6. Splits by `SUBJECT_ID`
7. Encodes the admission documents once and saves artifacts

## Outputs saved
Inside `EXP_DIR` this notebook saves:
- `metadata/`
  - `admission_docs_meta.parquet`
  - `train_adm_meta.parquet`
  - `val_adm_meta.parquet`
  - `test_adm_meta.parquet`
  - `manifest.json`
- `embeddings/`
  - `static_train.npz`
  - `static_val.npz`
  - `static_test.npz`

Notebook 2 starts from those saved artifacts.

## Note: as described in README, MIMIC-III is not included in this project, authorised PhysioNet access is required.

In [ ]:
# ============================================================
# CELL 1: Paths + experiment config
# - Here mortality -> heart failure and simplified scope
# ============================================================

from pathlib import Path

ADMISSIONS_PATH = "your path to ADMISSIONS.csv"
NOTEEVENTS_PATH = "your path to NOTEEVENTS.csv"
DIAGNOSES_PATH = "your path to DIAGNOSES_ICD.csv"
DICT_PATH = "your path to D_ICD_DIAGNOSES.csv"
MODEL_PATH = "your path to BiomedBERT-MIMIC-Contrastive"

MAX_HOSP_DAY = 0
DEV_SAMPLE = None
SEED = 42

HF_PREFIX = "428"

MIN_TEXT_LEN = 20
MAX_NOTES_PER_HADM = 60

CHUNKSIZE = 100_000

# These are currently optimized for 90GB of VRAM.
# Be wary of OOM
STATIC_BATCH_SIZE = 8196
STATIC_CHUNK_CHARS = 2000
STATIC_OVERLAP = 200
STATIC_MAX_CHUNKS = 16

FORCE_REBUILD_STATIC = False

EXPERIMENT_NAME = f"your heart disease experiment name"
ARTIFACT_ROOT = Path("your heart disease artifact root")
EXP_DIR = ARTIFACT_ROOT / EXPERIMENT_NAME

META_DIR = EXP_DIR / "metadata"
EMB_DIR = EXP_DIR / "embeddings"
MODEL_DIR = EXP_DIR / "models"

print("Experiment directory:")
print(EXP_DIR)


In [ ]:
# ============================================================
# CELL 2: Imports + reproducibility
# ============================================================

import json
import random
import re
from typing import List

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from IPython.display import display
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("DEVICE:", DEVICE)


In [ ]:
# ============================================================
# CELL 3: Helper functions
# ============================================================

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

for folder in [META_DIR, EMB_DIR, MODEL_DIR]:
    ensure_dir(folder)


def save_npz_array(path: Path, array: np.ndarray):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, array=array)


def load_npz_array(path: Path) -> np.ndarray:
    return np.load(path)["array"]


def save_table(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)


HF_EXPLICIT_TERMS = [
    r"\bheart failure\b",
    r"\bcongestive heart failure\b",
    r"\bcongestive cardiac failure\b",
    r"\bcardiac failure\b",
    r"\bheart disease\b",
    r"\bcongestive heart disease\b",
    r"\bchf\b",
    r"\badhf\b",
    r"\bdecompensated heart failure\b",
    r"\bacute decompensated heart failure\b",
    r"\bhfref\b",
    r"\bhfpef\b",
    r"\bheart-failure\b",
    r"\bheart-disease\b",
]
_HF_PAT = re.compile("|".join(HF_EXPLICIT_TERMS), flags=re.IGNORECASE)

def scrub_hf_terms(text_series: pd.Series) -> pd.Series:
    out = text_series.astype(str).str.replace(_HF_PAT, " ", regex=True)
    out = out.str.replace(r"\s+", " ", regex=True).str.strip()
    return out


def chunk_text_spread(text: str, chunk_chars: int, overlap: int, max_chunks: int) -> List[str]:
    text = str(text)
    if not text.strip():
        return []

    step = max(1, chunk_chars - overlap)
    chunks = [text[i:i + chunk_chars] for i in range(0, len(text), step)]

    if len(chunks) <= max_chunks:
        return chunks

    idx = np.linspace(0, len(chunks) - 1, max_chunks).astype(int)
    return [chunks[i] for i in idx]


def encode_docs_fast(
    texts,
    model: SentenceTransformer,
    batch_size: int,
    chunk_chars: int,
    overlap: int,
    max_chunks: int,
    pooling: str = "mean_max",
    normalize_after: bool = True
) -> np.ndarray:
    texts = list(texts)
    all_chunks = []
    spans = []

    for t in tqdm(texts, desc="Chunking long documents"):
        chunks = chunk_text_spread(
            t,
            chunk_chars=chunk_chars,
            overlap=overlap,
            max_chunks=max_chunks
        )
        start = len(all_chunks)
        all_chunks.extend(chunks)
        end = len(all_chunks)
        spans.append((start, end))

    print("Total chunks to encode:", len(all_chunks))

    if len(all_chunks) == 0:
        dim = model.get_sentence_embedding_dimension()
        out_dim = dim if pooling in ("mean", "max") else 2 * dim
        return np.zeros((len(texts), out_dim), dtype=np.float32)

    chunk_embs = model.encode(
        all_chunks,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)

    dim = chunk_embs.shape[1]
    out_dim = dim if pooling in ("mean", "max") else 2 * dim
    X = np.zeros((len(spans), out_dim), dtype=np.float32)

    for i, (s, e) in enumerate(spans):
        if s == e:
            continue

        embs = chunk_embs[s:e]

        if pooling == "mean":
            v = embs.mean(axis=0)
        elif pooling == "max":
            v = embs.max(axis=0)
        elif pooling == "mean_max":
            v = np.concatenate([embs.mean(axis=0), embs.max(axis=0)])
        else:
            raise ValueError("pooling must be 'mean', 'max', or 'mean_max'")

        if normalize_after:
            v = v / (np.linalg.norm(v) + 1e-12)

        X[i] = v.astype(np.float32)

    return X


In [ ]:
# ============================================================
# CELL 4: Load ADMISSIONS + DIAGNOSES and build the heart-failure label
# ============================================================

adm = pd.read_csv(
    ADMISSIONS_PATH,
    usecols=["SUBJECT_ID", "HADM_ID", "ADMITTIME"],
    parse_dates=["ADMITTIME"],
)
adm = adm.dropna(subset=["SUBJECT_ID", "HADM_ID", "ADMITTIME"]).copy()
adm["SUBJECT_ID"] = adm["SUBJECT_ID"].astype(int)
adm["HADM_ID"] = adm["HADM_ID"].astype(int)
adm["admit_date"] = adm["ADMITTIME"].dt.normalize()

d_icd = pd.read_csv(DICT_PATH)
d_icd["ICD9_CODE"] = d_icd["ICD9_CODE"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

hf_family = (
    d_icd[d_icd["ICD9_CODE"].str.startswith(HF_PREFIX, na=False)]
    .sort_values("ICD9_CODE")
    .reset_index(drop=True)
)

diag = pd.read_csv(
    DIAGNOSES_PATH,
    usecols=["SUBJECT_ID", "HADM_ID", "SEQ_NUM", "ICD9_CODE"],
    low_memory=False
)
diag = diag.dropna(subset=["SUBJECT_ID", "HADM_ID", "ICD9_CODE"]).copy()
diag["SUBJECT_ID"] = pd.to_numeric(diag["SUBJECT_ID"], errors="coerce")
diag["HADM_ID"] = pd.to_numeric(diag["HADM_ID"], errors="coerce")
diag = diag.dropna(subset=["SUBJECT_ID", "HADM_ID"]).copy()
diag["SUBJECT_ID"] = diag["SUBJECT_ID"].astype(int)
diag["HADM_ID"] = diag["HADM_ID"].astype(int)
diag["ICD9_CODE"] = diag["ICD9_CODE"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

hf_diag = diag[diag["ICD9_CODE"].str.startswith(HF_PREFIX, na=False)].copy()
hf_hadm = hf_diag[["HADM_ID"]].drop_duplicates().assign(label=1)

adm = adm.merge(hf_hadm, on="HADM_ID", how="left")
adm["label"] = adm["label"].fillna(0).astype(int)

print("ADMISSIONS shape:", adm.shape)
print("HF prevalence across all admissions:", adm["label"].mean())

print("\\n428* family from D_ICD_DIAGNOSES:")
display(hf_family)

print("\\nHF prevalence checks before note filtering:")
print(f"Total unique admissions in ADMISSIONS: {adm['HADM_ID'].nunique():,}")
print(f"Admissions with any {HF_PREFIX}* code: {adm['label'].sum():,}")
print(f"Admission prevalence: {100 * adm['label'].mean():.2f}%")

adm = adm[["SUBJECT_ID", "HADM_ID", "ADMITTIME", "admit_date", "label"]].copy()
display(adm.head())


In [ ]:
# ============================================================
# CELL 5: Stream NOTEEVENTS once and keep day-0 non-discharge notes
# ============================================================

adm_lookup = adm.set_index("HADM_ID")[["SUBJECT_ID", "ADMITTIME", "admit_date", "label"]]

note_rows = []
total_note_rows_seen = 0

usecols_notes = ["ROW_ID", "HADM_ID", "CHARTDATE", "CHARTTIME", "CATEGORY", "TEXT"]

for chunk in pd.read_csv(
    NOTEEVENTS_PATH,
    usecols=usecols_notes,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    total_note_rows_seen += len(chunk)

    chunk = chunk.dropna(subset=["HADM_ID", "TEXT"]).copy()
    if chunk.empty:
        continue

    chunk["HADM_ID"] = pd.to_numeric(chunk["HADM_ID"], errors="coerce")
    chunk = chunk.dropna(subset=["HADM_ID"]).copy()
    chunk["HADM_ID"] = chunk["HADM_ID"].astype(int)

    chunk = chunk[chunk["HADM_ID"].isin(adm_lookup.index)]
    if chunk.empty:
        continue

    chunk["CATEGORY"] = chunk["CATEGORY"].fillna("").astype(str)
    chunk["TEXT"] = chunk["TEXT"].astype(str)
    chunk = chunk[chunk["TEXT"].str.len() >= MIN_TEXT_LEN].copy()
    if chunk.empty:
        continue

    chunk["note_date"] = pd.to_datetime(chunk["CHARTDATE"], errors="coerce").dt.normalize()
    chunk["note_time"] = pd.to_datetime(chunk["CHARTTIME"], errors="coerce")
    chunk["note_time"] = chunk["note_time"].fillna(pd.to_datetime(chunk["CHARTDATE"], errors="coerce"))

    chunk = chunk.join(adm_lookup, on="HADM_ID", how="inner")
    if chunk.empty:
        continue

    chunk = chunk[~chunk["CATEGORY"].str.lower().eq("discharge summary")].copy()
    if chunk.empty:
        continue

    chunk = chunk.dropna(subset=["note_date"]).copy()
    chunk["hospital_day"] = (chunk["note_date"] - chunk["admit_date"]).dt.days
    chunk = chunk[(chunk["hospital_day"] >= 0) & (chunk["hospital_day"] <= MAX_HOSP_DAY)].copy()
    if chunk.empty:
        continue

    chunk["text_clean"] = scrub_hf_terms(chunk["TEXT"])
    chunk["text_clean"] = chunk["text_clean"].astype(str).str.strip()
    chunk = chunk[chunk["text_clean"].str.len() > 0].copy()
    if chunk.empty:
        continue

    note_rows.append(
        chunk[[
            "SUBJECT_ID", "HADM_ID", "admit_date", "note_date",
            "note_time", "hospital_day", "text_clean", "label"
        ]].copy()
    )

notes_df = pd.concat(note_rows, ignore_index=True) if len(note_rows) else pd.DataFrame(
    columns=["SUBJECT_ID", "HADM_ID", "admit_date", "note_date", "note_time", "hospital_day", "text_clean", "label"]
)

notes_df["SUBJECT_ID"] = pd.to_numeric(notes_df["SUBJECT_ID"], errors="coerce").fillna(-1).astype(int)
notes_df["HADM_ID"] = pd.to_numeric(notes_df["HADM_ID"], errors="coerce").fillna(-1).astype(int)
notes_df["label"] = pd.to_numeric(notes_df["label"], errors="coerce").fillna(0).astype(int)

print("Total streamed NOTEEVENTS rows seen:", total_note_rows_seen)
print("Day-0 non-discharge notes kept:", notes_df.shape)
display(notes_df.head())


In [ ]:
# ============================================================
# CELL 6: Build admission-level day-0 documents
# ============================================================

notes_df = notes_df.sort_values(["HADM_ID", "hospital_day", "note_time", "note_date"]).reset_index(drop=True)

notes_df = (
    notes_df.groupby("HADM_ID", group_keys=False)
            .head(MAX_NOTES_PER_HADM)
            .reset_index(drop=True)
)

print("Notes after per-admission cap:", notes_df.shape)

admission_docs = (
    notes_df.groupby(["HADM_ID", "SUBJECT_ID"], as_index=False)
            .agg(
                admit_date=("admit_date", "first"),
                label=("label", "first"),
                note_count=("text_clean", "size"),
                text_window=("text_clean", lambda s: "\n\n----- NOTE -----\n\n".join(s.dropna()))
            )
)

admission_docs["text_window_len"] = admission_docs["text_window"].astype(str).str.len()

if DEV_SAMPLE is not None:
    if DEV_SAMPLE > len(admission_docs):
        raise ValueError(f"DEV_SAMPLE={DEV_SAMPLE} is larger than available admissions {len(admission_docs)}")

    admission_docs = admission_docs.sample(DEV_SAMPLE, random_state=SEED).reset_index(drop=True)
    sampled_hadm_ids = set(admission_docs["HADM_ID"].tolist())
    notes_df = notes_df[notes_df["HADM_ID"].isin(sampled_hadm_ids)].reset_index(drop=True)

print("Admissions in day-0 note cohort:", admission_docs.shape)
print("HF prevalence after note filtering:", admission_docs["label"].mean())
display(admission_docs.head())

if len(admission_docs) == 0:
    raise ValueError("No admission-level documents were built. Check the day window and note filters.")


In [ ]:
# ============================================================
# CELL 7: Subject-level train / val / test splits
# ============================================================

subjects = admission_docs["SUBJECT_ID"].drop_duplicates().values
train_subj, temp_subj = train_test_split(subjects, test_size=0.30, random_state=SEED)
val_subj, test_subj = train_test_split(temp_subj, test_size=0.50, random_state=SEED)

split_map = {}
for s in train_subj:
    split_map[int(s)] = "train"
for s in val_subj:
    split_map[int(s)] = "val"
for s in test_subj:
    split_map[int(s)] = "test"

admission_docs["split"] = admission_docs["SUBJECT_ID"].map(split_map)

train_adm = admission_docs[admission_docs["split"] == "train"].reset_index(drop=True)
val_adm   = admission_docs[admission_docs["split"] == "val"].reset_index(drop=True)
test_adm  = admission_docs[admission_docs["split"] == "test"].reset_index(drop=True)

print("Static split sizes:", train_adm.shape, val_adm.shape, test_adm.shape)
print("\\nHF prevalence by split:")
print(admission_docs.groupby("split")["label"].mean())

display(train_adm.head())


In [ ]:
# ============================================================
# CELL 8: Save metadata tables + manifest
# ============================================================

save_table(
    admission_docs[["HADM_ID", "SUBJECT_ID", "admit_date", "label", "split", "note_count", "text_window_len"]].copy(),
    META_DIR / "admission_docs_meta.parquet"
)
save_table(
    train_adm[["HADM_ID", "SUBJECT_ID", "admit_date", "label", "split", "note_count", "text_window_len"]].copy(),
    META_DIR / "train_adm_meta.parquet"
)
save_table(
    val_adm[["HADM_ID", "SUBJECT_ID", "admit_date", "label", "split", "note_count", "text_window_len"]].copy(),
    META_DIR / "val_adm_meta.parquet"
)
save_table(
    test_adm[["HADM_ID", "SUBJECT_ID", "admit_date", "label", "split", "note_count", "text_window_len"]].copy(),
    META_DIR / "test_adm_meta.parquet"
)

manifest = {
    "experiment_name": EXPERIMENT_NAME,
    "task_name": "heart_failure_present_during_admission",
    "label_definition": f"1 if any ICD-9 code starts with {HF_PREFIX}; else 0",
    "hf_prefix": HF_PREFIX,
    "max_hosp_day": int(MAX_HOSP_DAY),
    "seed": int(SEED),
    "model_path": MODEL_PATH,
    "static_batch_size": int(STATIC_BATCH_SIZE),
    "notes_cap_per_hadm": int(MAX_NOTES_PER_HADM),
    "dev_sample": None if DEV_SAMPLE is None else int(DEV_SAMPLE),
    "retrieval_used": False,
    "risk_features_used": False,
    "scrubber": "explicit heart-failure / heart-disease terms removed before embedding",
}

with open(META_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Metadata + manifest saved.")


In [ ]:
# ============================================================
# CELL 9: Load the encoder
# ============================================================

print("Loading encoder from:", MODEL_PATH)
encoder = SentenceTransformer(MODEL_PATH, device=DEVICE)

if DEVICE == "cuda":
    try:
        encoder.half()
        print("Encoder switched to fp16 for inference.")
    except Exception as e:
        print("Could not switch encoder to fp16:", e)

print("Sentence embedding dimension:", encoder.get_sentence_embedding_dimension())


In [ ]:
# ============================================================
# CELL 10: Encode static admission docs (with caching)
# ============================================================

static_train_path = EMB_DIR / "static_train.npz"
static_val_path   = EMB_DIR / "static_val.npz"
static_test_path  = EMB_DIR / "static_test.npz"

if (
    static_train_path.exists() and static_val_path.exists() and static_test_path.exists()
    and not FORCE_REBUILD_STATIC
):
    print("Loading cached static embeddings from Drive...")
    X_train_static = load_npz_array(static_train_path)
    X_val_static   = load_npz_array(static_val_path)
    X_test_static  = load_npz_array(static_test_path)
else:
    print("Encoding static admission documents...")
    X_train_static = encode_docs_fast(
        train_adm["text_window"],
        encoder,
        batch_size=STATIC_BATCH_SIZE,
        chunk_chars=STATIC_CHUNK_CHARS,
        overlap=STATIC_OVERLAP,
        max_chunks=STATIC_MAX_CHUNKS,
        pooling="mean_max",
    )
    X_val_static = encode_docs_fast(
        val_adm["text_window"],
        encoder,
        batch_size=STATIC_BATCH_SIZE,
        chunk_chars=STATIC_CHUNK_CHARS,
        overlap=STATIC_OVERLAP,
        max_chunks=STATIC_MAX_CHUNKS,
        pooling="mean_max",
    )
    X_test_static = encode_docs_fast(
        test_adm["text_window"],
        encoder,
        batch_size=STATIC_BATCH_SIZE,
        chunk_chars=STATIC_CHUNK_CHARS,
        overlap=STATIC_OVERLAP,
        max_chunks=STATIC_MAX_CHUNKS,
        pooling="mean_max",
    )

    save_npz_array(static_train_path, X_train_static)
    save_npz_array(static_val_path, X_val_static)
    save_npz_array(static_test_path, X_test_static)

print("X_train_static:", X_train_static.shape)
print("X_val_static:", X_val_static.shape)
print("X_test_static:", X_test_static.shape)


In [ ]:
# ============================================================
# CELL 11: Final artifact summary
# ============================================================

print("Saved experiment folder:")
print(EXP_DIR)

print("\nMetadata files:")
for p in sorted(META_DIR.glob("*")):
    print(" -", p.name)

print("\nEmbedding files:")
for p in sorted(EMB_DIR.glob("*")):
    print(" -", p.name)

print("\nNotebook 1 complete.")
print("You can now open HeartDisease_Downstream_Notebook2 and load this same EXP_DIR without re-encoding.")
